# LSTM · 13 Handover

Prints the row for the group's comparison table and lists the result files. **Saves:** nothing.

**Needs the trained LSTM** saved by notebook 08 (`results/lstm_model.keras`). If it is not found, it is trained automatically first (about 15 seconds, same settings and seed, so the same result).

*Notebook 13 of 14 · Previous: 12 Robustness Check · Next: 14 Optional Extension*

## Setup (the same in every notebook)

Each notebook in this project is self-contained. The cells below define the group settings and only the helper functions this notebook uses. They are identical in every notebook and nothing is imported from another file, so the notebook also runs on its own, for example in Colab.

In [ ]:
# ---- Imports and dataset location ----------------------------------------------------------------
import sys, re, json, time, pathlib, textwrap
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tensorflow import keras
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)

layers = keras.layers

# Locate dataset and split files locally
DATA_NAME = "deceptive-opinion.csv"
SPLIT_NAME = "split_assignment.csv"
here = pathlib.Path.cwd().resolve()
search_roots = [here, here.parent, here.parent.parent]

data_candidates = [r / "dataset" / DATA_NAME for r in search_roots] + [r / DATA_NAME for r in search_roots]
DATA_PATH = next((p for p in data_candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"Cannot find {DATA_NAME}. Ensure dataset/{DATA_NAME} exists.")

split_candidates = [r / "shared" / SPLIT_NAME for r in search_roots] + [r / SPLIT_NAME for r in search_roots]
SPLIT_PATH = next((p for p in split_candidates if p.exists()), None)
if SPLIT_PATH is None:
    raise FileNotFoundError(f"Cannot find {SPLIT_NAME}. Ensure shared/{SPLIT_NAME} exists.")

# Root folder for LSTM
LSTM_ROOT = next((p for p in [here, *here.parents] if (p / "01_Setup").is_dir() or (p / "LSTM_model.ipynb").exists()), here)
RESULTS = LSTM_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

print("Dataset:", DATA_PATH)
print("Split file:", SPLIT_PATH)
print("Results folder:", RESULTS)


In [ ]:
# ---- Group settings: identical for Simple RNN, LSTM, GRU and 1D CNN ------------------------------
SEED = 42

# Labels: this dataset has no star rating, so the `polarity` column is the label.
LABEL_COLUMN = "polarity"
LABEL_MAP = {"negative": 0, "positive": 1}

# Text -> integer sequences
VOCAB_SIZE = 5000          # total ids, including <PAD>=0 and <OOV>=1
MAX_LEN = 300              # tokens per review after cleaning
PAD_TOKEN, OOV_TOKEN = "<PAD>", "<OOV>"
PADDING = "pre"            # zeros in front, so a recurrent layer ends on real words
TRUNCATING = "post"        # long reviews keep their first MAX_LEN tokens

# Training
EMBED_DIM = 64
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_EPOCHS = 30
PATIENCE = 5               # EarlyStopping patience, monitoring validation loss
MONITOR = "val_loss"
THRESHOLD = 0.5            # probability cut-off used for every model's test metrics

# Fixed 80/10/10 split loaded directly from shared/split_assignment.csv
split_df = pd.read_csv(SPLIT_PATH)
VAL_IDS = set(split_df.loc[split_df["split"] == "val", "row_id"])
TEST_IDS = set(split_df.loc[split_df["split"] == "test", "row_id"])

keras.utils.set_random_seed(SEED)          # seeds Python, NumPy and TensorFlow
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| seed", SEED)
print("Devices:", [d.device_type for d in tf.config.list_physical_devices()])
print(f"Loaded fixed split: {len(VAL_IDS)} validation rows, {len(TEST_IDS)} test rows.")


In [ ]:
# ---- Data pipeline: load -> label -> de-duplicate -> split -> clean -> vocabulary -> pad ----------
# Leakage rule: the vocabulary and the class weights are built from TRAINING rows only.
# Validation and test rows are only ever encoded with the training vocabulary.

def load_labelled_data() -> pd.DataFrame:
    """Read the CSV, map polarity -> 0/1 and drop exact duplicate reviews.

    Duplicates must go before splitting, otherwise a copy of a training review could land in the
    test set and the test set would no longer be unseen.
    """
    df = pd.read_csv(DATA_PATH)
    df["row_id"] = np.arange(len(df))                   # 0-based row number in the CSV
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df[LABEL_COLUMN].map(LABEL_MAP)
    assert df["label"].notna().all(), "unexpected label value"
    df["label"] = df["label"].astype(int)
    df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
    return df[["row_id", "text", "label", "hotel", "deceptive", "source"]]


def assign_split(df: pd.DataFrame) -> np.ndarray:
    """'train' / 'val' / 'test' for every row, from the fixed id lists above."""
    if not (VAL_IDS | TEST_IDS) <= set(df["row_id"]):
        raise ValueError("The fixed validation/test ids do not match this dataset. Was the CSV changed?")
    return np.where(df["row_id"].isin(TEST_IDS), "test",
                    np.where(df["row_id"].isin(VAL_IDS), "val", "train"))


# Text cleaning: identical for all models; a pure per-review function, nothing is fitted.
_HTML = re.compile(r"<[^>]+>")
_URL = re.compile(r"(https?://\S+|www\.\S+)")
_CONTRACTIONS = [                       # keep negation: "didn't" -> "did not", not "didn t"
    (r"\bwon't\b", "will not"), (r"\bcan't\b", "can not"), (r"\bcannot\b", "can not"),
    (r"n't\b", " not"), (r"'re\b", " are"), (r"'ve\b", " have"), (r"'ll\b", " will"),
    (r"'d\b", " would"), (r"'m\b", " am"), (r"'s\b", ""),
]
_NON_LETTER = re.compile(r"[^a-z\s]")
_SPACES = re.compile(r"\s+")


def clean_text(text: str) -> str:
    """lowercase -> strip HTML -> strip URLs -> expand contractions -> keep letters only.

    Stop-words are deliberately NOT removed: words such as "not", "no" and "never" flip the sentiment.
    """
    text = text.lower().replace("’", "'")
    text = _HTML.sub(" ", text)
    text = _URL.sub(" ", text)
    for pattern, replacement in _CONTRACTIONS:
        text = re.sub(pattern, replacement, text)
    text = _NON_LETTER.sub(" ", text)           # digits, punctuation, symbols
    return _SPACES.sub(" ", text).strip()


def build_vocab(train_token_lists, vocab_size: int = VOCAB_SIZE) -> dict:
    """Most frequent TRAINING words -> ids. 0 = <PAD>, 1 = <OOV>, real words start at 2."""
    counts = Counter(tok for tokens in train_token_lists for tok in tokens)
    words = [w for w, _ in counts.most_common(vocab_size - 2)]
    vocab = {PAD_TOKEN: 0, OOV_TOKEN: 1}
    vocab.update({w: i + 2 for i, w in enumerate(words)})
    return vocab


def encode(tokens, vocab: dict) -> list:
    oov = vocab[OOV_TOKEN]
    return [vocab.get(t, oov) for t in tokens]


def decode(ids, vocab: dict, skip_pad: bool = True) -> list:
    """Inverse of `encode`: ids -> words. Unknown words come back as <OOV> (the original is lost)."""
    inverse = {i: w for w, i in vocab.items()}
    return [inverse[int(i)] for i in ids if not (skip_pad and int(i) == 0)]


def pad(sequences, max_len: int = MAX_LEN) -> np.ndarray:
    out = np.zeros((len(sequences), max_len), dtype="int32")
    for i, seq in enumerate(sequences):
        seq = seq[:max_len] if TRUNCATING == "post" else seq[-max_len:]
        if PADDING == "pre":
            out[i, max_len - len(seq):] = seq
        else:
            out[i, :len(seq)] = seq
    return out


def compute_class_weights(y_train) -> dict:
    """weight_c = n_samples / (n_classes * n_samples_in_class_c)   (same as sklearn 'balanced')."""
    classes, counts = np.unique(y_train, return_counts=True)
    return {int(c): float(len(y_train) / (len(classes) * n)) for c, n in zip(classes, counts)}


@dataclass
class Data:
    X_train: np.ndarray
    y_train: np.ndarray
    X_val: np.ndarray
    y_val: np.ndarray
    X_test: np.ndarray
    y_test: np.ndarray
    vocab: dict
    class_weight: dict
    train_df: pd.DataFrame     # raw + cleaned text, kept for EDA / error analysis
    val_df: pd.DataFrame
    test_df: pd.DataFrame


def prepare_data() -> Data:
    """Everything a model needs, in one call."""
    df = load_labelled_data()
    df["split"] = assign_split(df)
    df["clean"] = df["text"].map(clean_text)
    df["tokens"] = df["clean"].str.split()

    parts = {name: g.reset_index(drop=True) for name, g in df.groupby("split")}
    vocab = build_vocab(parts["train"]["tokens"].tolist())          # <- TRAIN ONLY

    def to_xy(part: pd.DataFrame):
        seqs = [encode(tokens, vocab) for tokens in part["tokens"]]
        return pad(seqs), part["label"].to_numpy(dtype="int32")

    X_train, y_train = to_xy(parts["train"])
    X_val, y_val = to_xy(parts["val"])
    X_test, y_test = to_xy(parts["test"])
    return Data(X_train, y_train, X_val, y_val, X_test, y_test, vocab,
                compute_class_weights(y_train), parts["train"], parts["val"], parts["test"])

In [ ]:
# ---- Evaluation helpers: the same metrics and result format for all four models ------------------
# Positive class (label 1) is the class used for precision / recall / F1.

def compute_metrics(y_true, y_prob, threshold: float = THRESHOLD) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).ravel()
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        # per-class view, needed for the error analysis
        "negative_precision": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "negative_recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "negative_f1": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def bootstrap_ci(y_true, y_prob, n_boot: int = 2000, seed: int = SEED) -> dict:
    """95% bootstrap interval for accuracy / F1 / ROC-AUC on the test set.
    It only describes how uncertain the (small) test set makes the numbers. It is NOT used to tune anything."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).ravel()
    rng = np.random.default_rng(seed)
    out = {"accuracy": [], "f1": [], "roc_auc": []}
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        m = compute_metrics(y_true[idx], y_prob[idx])
        for k in out:
            out[k].append(m[k])
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in out.items()}


def save_result(path, model_name: str, test_metrics: dict, *, training_time_sec: float,
                trainable_params: int, total_params: int, epochs_run: int, best_epoch: int,
                extra: dict = None) -> dict:
    """One JSON file per model with the same keys for every model, for the group's comparison table."""
    row = {
        "model": model_name,
        "accuracy": test_metrics["accuracy"], "precision": test_metrics["precision"],
        "recall": test_metrics["recall"], "f1": test_metrics["f1"], "roc_auc": test_metrics["roc_auc"],
        "training_time_sec": training_time_sec,
        "trainable_params": trainable_params, "total_params": total_params,
        "epochs_run": epochs_run, "best_epoch": best_epoch,
        "seed": SEED, "max_len": MAX_LEN, "vocab_size": VOCAB_SIZE,
        "embed_dim": EMBED_DIM, "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "threshold": THRESHOLD,
        "confusion_matrix": {k: test_metrics[k] for k in ("tn", "fp", "fn", "tp")},
    }
    if extra:
        row.update(extra)
    with open(path, "w") as f:
        json.dump(row, f, indent=2)
    return row

In [ ]:
# ---- Model helpers (the same functions as in notebooks 07 and 08) ----
def build_lstm():
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(VOCAB_SIZE, EMBED_DIM),
        layers.LSTM(64),
        layers.Dropout(0.5),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ], name="lstm_sentiment")

def n_params(weights):
    return int(sum(np.prod(w.shape) for w in weights))

def train_lstm(seed, verbose=2):
    keras.utils.set_random_seed(seed)
    m = build_lstm()
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
              loss="binary_crossentropy", metrics=["accuracy"])
    early_stop = keras.callbacks.EarlyStopping(monitor=MONITOR, patience=PATIENCE,
                                               restore_best_weights=True)
    start = time.perf_counter()
    hist = m.fit(data.X_train, data.y_train,
                 validation_data=(data.X_val, data.y_val),          # validation only, never test
                 epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                 class_weight=data.class_weight, callbacks=[early_stop], verbose=verbose)
    return m, hist.history, time.perf_counter() - start

def save_training_outputs(model, history, train_time, epochs_run, best_epoch):
    # what the later notebooks need to continue without retraining
    model.save(RESULTS / "lstm_model.keras")
    (pd.DataFrame(history).rename_axis("epoch").reset_index().assign(epoch=lambda d: d.epoch + 1)
       .to_csv(RESULTS / "lstm_history.csv", index=False))
    (RESULTS / "lstm_training_info.json").write_text(json.dumps(
        {"train_time_sec": train_time, "epochs_run": epochs_run, "best_epoch": best_epoch}))
def load_or_train_main():
    # Returns (model, history, train_time, epochs_run, best_epoch) for the main LSTM (seed 42).
    # It uses the files saved by notebook 08 when they exist. Otherwise it trains the LSTM here
    # (about 15 seconds, same settings and seed, so the result is the same). Needs `data` to exist.
    paths = [RESULTS / n for n in ("lstm_model.keras", "lstm_history.csv", "lstm_training_info.json")]
    if all(p.exists() for p in paths):
        info = json.loads(paths[2].read_text())
        history = pd.read_csv(paths[1]).drop(columns="epoch").to_dict("list")
        print("Loaded the trained LSTM saved by notebook 08.")
        return keras.models.load_model(paths[0]), history, info["train_time_sec"], info["epochs_run"], info["best_epoch"]
    print("No saved LSTM found in results/, so it is trained here (same settings as notebook 08)...")
    model, history, train_time = train_lstm(SEED, verbose=0)
    epochs_run, best_epoch = len(history["loss"]), int(np.argmin(history["val_loss"])) + 1
    save_training_outputs(model, history, train_time, epochs_run, best_epoch)
    return model, history, train_time, epochs_run, best_epoch

### Load what this notebook needs

In [ ]:
data = prepare_data()
model, history, train_time, epochs_run, best_epoch = load_or_train_main()
y_prob = model.predict(data.X_test, verbose=0).ravel()     # test predictions, only used to describe the results
test = compute_metrics(data.y_test, y_prob)

## Hand-over to the group

Files written to `lstm/results/`:

| File | Use |
|---|---|
| `lstm_metrics.json` | The LSTM row for the comparison table (same schema for all models) |
| `lstm_test_predictions.csv` | Per-review test predictions, so the group can compare models review by review |
| `lstm_history.csv` | Epoch-by-epoch training and validation loss and accuracy |
| `lstm_accuracy_curve.png`, `lstm_loss_curve.png`, `lstm_confusion_matrix.png`, `lstm_roc_curve.png` | The four required figures |
| `lstm_probability_hist.png` | Extra figure: model confidence on the test set |
| `lstm_audit_length_outliers.png`, `lstm_eda_class_balance.png`, `lstm_eda_distinctive_words.png`, `lstm_eda_review_length.png` | Data audit and EDA figures |
| `lstm_seed_robustness.csv` | Run-to-run variation |
| `lstm_model.keras` | The trained model |
| `lstm_short_model.keras`, `lstm_short_extension_metrics.json` | Optional notebook 14 extension model and its metrics (not part of the four-model comparison) |

**Comparison table row:** paste the row printed in notebook 10. The overall best model can only be named after all four models have been evaluated under these same conditions.
**Training time** is only comparable between models if all four are timed on the same hardware.

In [ ]:
trainable = n_params(model.trainable_weights)
print("Row for the group comparison table:\n")
print("| Model | Accuracy | Precision | Recall | F1-score | ROC-AUC | Training Time | Parameters |")
print("|---|---|---|---|---|---|---|---|")
print(f"| LSTM | {test['accuracy']:.4f} | {test['precision']:.4f} | {test['recall']:.4f} | {test['f1']:.4f} "
      f"| {test['roc_auc']:.4f} | {train_time:.1f} s | {trainable:,} |")

print("\nFiles in", RESULTS)
for f in sorted(RESULTS.iterdir()):
    print(" ", f.name)

---
*Notebook 13 of 14 · Previous: 12 Robustness Check · Next: 14 Optional Extension*